<a href="https://colab.research.google.com/github/doomchitdoomchit/OD_DAY_SIDO_SEOULGU/blob/master/notebook/%EC%A7%80%EC%97%AD%EB%B3%84_%EC%83%81%ED%92%88_%EB%AC%BC%EB%8F%99%EB%9F%89_%ED%86%B5%EA%B3%84_%EB%8D%B0%EC%9D%B4%ED%84%B0_ADDlog_POSTHoli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import holidays

import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
drive_path = '/content/drive/My Drive/Colab Notebooks/지역별 상품 물동량 통계 데이터'
data_name = 'seoulgu.parquet'

In [ ]:
# data load
lazy_df = pl.scan_parquet(os.path.join(drive_path, data_name))

In [ ]:
all_cols = lazy_df.columns
cat_cols = [c for c in all_cols if c.startswith('CAT_VOL_')]

/tmp/ipykernel_3118/2883817882.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_cols = lazy_df.columns


In [ ]:
# 날짜 타입 변환 및 11개 카테고리 수평 합산 (total_vol)
lazy_processed = lazy_df.with_columns([
    pl.col('DELIVERY_YMD').cast(pl.Date).alias('DELIVERY_YMD'),
    pl.sum_horizontal(cat_cols).alias('total_vol') ])


In [ ]:
# (2) 자치구 및 일자별 집계 및 정렬
lazy_daily = (
    lazy_processed.group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM']).agg(
        pl.col('total_vol').sum()
        ).sort(['RECEIVER_SIGUNGU_NM', 'DELIVERY_YMD'])
    )


In [ ]:
# (3) 시계열 피처 생성 (캘린더 + 시차 Lag + 이동 통계량 Rolling)
lags = [7, 14, 21, 28]
windows = [7, 14, 28]

# Lag 피처 표현식
lag_exprs = [
    pl.col('total_vol').shift(lag).over('RECEIVER_SIGUNGU_NM').alias(f'vol_lag_{lag}')
    for lag in lags
]

# Rolling 피처 표현식 (Data Leakage 방지를 위해 Shift 7 적용)
rolling_exprs = []
for w in windows:
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_mean(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_mean_{w}'))
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_std(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_std_{w}'))


In [ ]:
def add_holiday_backlog_features(df_daily: pl.DataFrame, start_year: int = 2019, end_year: int = 2023) -> pl.DataFrame:
    # 1. 대한민국 관공서 공휴일 추출 (holidays 라이브러리)
    kr_holidays = set(holidays.KR(years=range(start_year, end_year + 1)).keys())

    # 2. 휴무일(is_off_day) 정의: 일요일(7) 또는 법정 공휴일
    df_feat = df_daily.with_columns([
        pl.col('DELIVERY_YMD').map_elements(
            lambda d: 1 if (d.weekday() == 7 or d in kr_holidays) else 0,
            return_dtype=pl.Int64
        ).alias('is_off_day')
    ])

    # 3. 연속 휴무 구간(Group ID) 식별
    df_feat = df_feat.with_columns([
        (pl.col('is_off_day') != pl.col('is_off_day').shift(1))
        .fill_null(True)
        .cum_sum()
        .alias('off_block_id')
    ])

    # 4. 휴무 구간별 연속 휴무 일수 계산
    off_block_stats = (
        df_feat.filter(pl.col('is_off_day') == 1)
        .group_by('off_block_id')
        .agg(pl.col('is_off_day').sum().alias('consecutive_off_days'))
    )

    # 5. 원본 결합 및 전후(Pre/Post) 연휴 일수 가져오기
    df_joined = df_feat.join(off_block_stats, on='off_block_id', how='left') \
                       .with_columns(pl.col('consecutive_off_days').fill_null(0))

    df_final = df_joined.with_columns([
        # 직전 연휴 일수 (어제 기준 shift 1) -> Post-Holiday 계산용
        pl.col('consecutive_off_days').shift(1).over('RECEIVER_SIGUNGU_NM').fill_null(0).alias('prev_off_days'),

        # ★ 다가올 연휴 일수 (내일 기준 shift -1) -> Pre-Holiday 계산용
        pl.col('consecutive_off_days').shift(-1).over('RECEIVER_SIGUNGU_NM').fill_null(0).alias('next_off_days')
    ]).with_columns([
        # --- [Post-Holiday: 연휴 직후 이월 피처] ---
        # D+1 (연휴 직후 첫 영업일 - 이월 물량 폭증)
        ((pl.col('is_off_day') == 0) & (pl.col('prev_off_days') > 0)).cast(pl.Int64).alias('is_post_holiday_d1'),
        # D+2 (연휴 직후 둘째 영업일)
        ((pl.col('is_off_day') == 0) & (pl.col('prev_off_days').shift(1).over('RECEIVER_SIGUNGU_NM') > 0)).cast(pl.Int64).alias('is_post_holiday_d2'),
        # 연휴 직후 이월 폭발력 스코어
        pl.when((pl.col('is_off_day') == 0) & (pl.col('prev_off_days') > 0))
          .then(pl.col('prev_off_days'))
          .otherwise(0).alias('backlog_impact_score'),

        # --- [Pre-Holiday: 연휴 직전 쏠림 피처 — ⭐ 신규 추가] ---
        # D-1 (연휴 시작 직전 1일차 - 배송 마감 출하 피크)
        ((pl.col('is_off_day') == 0) & (pl.col('next_off_days')> 0)).cast(pl.Int64).alias('is_pre_holiday_d1'),
        # D-2 (연휴 시작 직전 2일차)
        ((pl.col('is_off_day') == 0) & (pl.col('next_off_days').shift(-1).over('RECEIVER_SIGUNGU_NM')> 0)).cast(pl.Int64).alias('is_pre_holiday_d2'),
        # 연휴 직전 사전 집하 쏠림 스코어 (D-1 여부 * 다가올 연휴 일수)
        pl.when((pl.col('is_off_day') == 0) & (pl.col('next_off_days')> 0))
          .then(pl.col('next_off_days'))
          .otherwise(0).alias('pre_holiday_impact_score')
    ])

    return df_final

In [ ]:
add_holiday = add_holiday_backlog_features(lazy_daily)

In [ ]:
# 지연 피처 엔지니어링 쿼리 결합
lazy_featured = add_holiday.with_columns([
    pl.col('DELIVERY_YMD').dt.year().alias('year'),
    pl.col('DELIVERY_YMD').dt.month().alias('month'),
    pl.col('DELIVERY_YMD').dt.day().alias('day'),
    pl.col('DELIVERY_YMD').dt.weekday().alias('dayofweek'),
    pl.col('DELIVERY_YMD').dt.weekday().is_in([6, 7]).cast(pl.Int8).alias('is_weekend'),
    pl.col('RECEIVER_SIGUNGU_NM').cast(pl.Categorical)
    ] + lag_exprs + rolling_exprs).drop_nulls()

In [ ]:
print("Lazy Query Plan 최적화 및 연산 실행 중...")
df_processed = lazy_featured.collect()
print(f"연산 완료된 데이터 Shape: {df_processed.shape}")

Lazy Query Plan 최적화 및 연산 실행 중...
연산 완료된 데이터 Shape: (44754, 29)


In [ ]:
max_date = df_processed['DELIVERY_YMD'].max()
split_date = max_date - pl.duration(days=90)

# 최근 3개월 검증
train_df = df_processed.filter(pl.col('DELIVERY_YMD') < split_date)
test_df = df_processed.filter(pl.col('DELIVERY_YMD') >= split_date)
features = [ 'RECEIVER_SIGUNGU_NM', 'month', 'day', 'dayofweek', 'is_weekend',
    # 2\. 연휴 직전 쏠림 (Pre-Holiday) 피처 (3개)
    'is_pre_holiday_d1',        # 연휴 직전 1일 차 (배송 마감 피크)
    'is_pre_holiday_d2',        # 연휴 직전 2일 차
    'pre_holiday_impact_score', # 사전 출하 쏠림 스코어 (D-1 여부 × 연휴 일수)

    # 3. 연휴 직후 이월 (Post-Holiday) 피처 (3개)
    'is_post_holiday_d1',       # 연휴 직후 1일 차 (터미널 이월 폭증 피크)
    'is_post_holiday_d2',       # 연휴 직후 2일 차 (잔여 이월 처리일)
    'backlog_impact_score',     # 이월 폭발력 스코어 (D+1 여부 × 연휴 일수)

    # 4. 시차 피처 (Lag Features - 최소 Lag 7 이상으로 Data Leakage 차단) (4개)
    'vol_lag_7', 'vol_lag_14', 'vol_lag_21', 'vol_lag_28',

    # 5. 이동 통계 피처 (Rolling Features - Shift 7 기준) (6개)
    'rolling_mean_7', 'rolling_std_7',
    'rolling_mean_14', 'rolling_std_14',
    'rolling_mean_28', 'rolling_std_28'
]
target = 'total_vol'

# LightGBM 입력을 위한 Pandas 변환 (Arrow 메모리 0-Copy 기반 변환)
X_train = train_df.select(features).to_pandas()
y_train = train_df.select(target).to_pandas().squeeze()
X_test = test_df.select(features).to_pandas()
y_test = test_df.select(target).to_pandas().squeeze()

In [ ]:
model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
###########################################################
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
###########################################################
model.fit(
    X_train, y_train_log,
    eval_set=[(X_test, y_test_log)],
    categorical_feature=['RECEIVER_SIGUNGU_NM'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

preds = model.predict(X_test)
###########################################################

preds = np.expm1(preds)
# y_train_log = np.log1p(y_train)
# y_test_log = np.log1p(y_test)
###########################################################

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
wape = (np.sum(np.abs(y_test - preds)) / np.sum(y_test)) * 100

print(f"\n=== LazyFrame 처리 기반 LightGBM 최종 백테스팅 결과 ===")
print(f"검증 기간: {split_date} ~ {max_date}")
print(f"RMSE : {rmse:,.2f} 건")
print(f"MAE  : {mae:,.2f} 건")
print(f"MAPE : {mape:.2f} %")
print(f"WAPE : {wape:.2f} %")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008718 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2649
[LightGBM] [Info] Number of data points in the train set: 42479, number of used features: 21
[LightGBM] [Info] Start training from score 9.997141

=== LazyFrame 처리 기반 LightGBM 최종 백테스팅 결과 ===
검증 기간: [(2023-12-30) - (90d.alias("duration"))] ~ 2023-12-30
RMSE : 9,368.76 건
MAE  : 5,581.96 건
MAPE : 61.99 %
WAPE : 15.76 %


In [ ]:
df_importance = pd.DataFrame({
    'feature': X_train.columns,
    'split': model.booster_.feature_importance(importance_type='split'),
    'gain': model.booster_.feature_importance(importance_type='gain')
}).sort_values(by='gain', ascending=False).reset_index(drop=True)

print(df_importance.head(10), '\n', df_importance.tail(10))

          feature  split           gain
0       dayofweek   1420  589659.223418
1             day   2251  118371.030308
2           month   2159   89840.220366
3      vol_lag_21    705   59459.231457
4  rolling_mean_7   1292   56871.964445
5      vol_lag_28    601   44107.425407
6       vol_lag_7    887   33598.083489
7  rolling_std_14    568   33481.282771
8      vol_lag_14    517   18671.547969
9  rolling_std_28    385   14861.606597 
                      feature  split         gain
11           rolling_mean_28    282  9686.064506
12         is_pre_holiday_d1    175  6629.618712
13         is_pre_holiday_d2    141  6448.132649
14  pre_holiday_impact_score     93  5525.872130
15        is_post_holiday_d2    126  3855.066948
16             rolling_std_7    154  2720.206887
17           rolling_mean_14    122  1732.060308
18      backlog_impact_score     79  1621.706035
19       RECEIVER_SIGUNGU_NM    109   649.599377
20                is_weekend      0     0.000000
